In [ ]:
import os 
os.environ["GEMINI_API_KEY"] = "key"

In [ ]:
# %pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

  Using cached pypdf-6.12.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached pybase64-1.4.3-cp311-cp311-win_amd64.whl.metadata (9.1 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached mmh3-5.2.1-cp311-cp311-win_amd64.whl.metadata (15 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB 435.7 kB/s eta 0:00:54
   --------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [5]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [6]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [11]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory="my_chroma_db",
    collection_name="sample"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4159.53it/s]


In [12]:
# Add documents
vector_store.add_documents(docs)

['8af2d290-c2fc-4f7c-984b-4286b321d1f0',
 '239cd839-9e73-4dd7-b268-b77a47bdc413',
 '2cc46849-0f0c-437e-ac9e-581cb4fcabf0',
 'b8c80b6d-283e-4d04-903f-cea8b1babd4c',
 '3e705696-6bc9-4001-9dd9-6b41fe724af0']

In [13]:
# view documents 
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['8af2d290-c2fc-4f7c-984b-4286b321d1f0',
  '239cd839-9e73-4dd7-b268-b77a47bdc413',
  '2cc46849-0f0c-437e-ac9e-581cb4fcabf0',
  'b8c80b6d-283e-4d04-903f-cea8b1babd4c',
  '3e705696-6bc9-4001-9dd9-6b41fe724af0'],
 'embeddings': array([[ 0.00994727,  0.06914337, -0.05147117, ..., -0.03543334,
          0.01284807,  0.0124829 ],
        [ 0.00127744,  0.03129853, -0.02375376, ..., -0.00518358,
         -0.03280612,  0.02737714],
        [-0.10265914,  0.02650812,  0.02271502, ..., -0.03359744,
         -0.07984945, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494375, ..., -0.10995811,
          0.00572558,  0.09915376],
        [ 0.01873978,  0.04382848, -0.04304255, ..., -0.07801619,
         -0.0784068 , -0.00304187]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [14]:
# search documents
vector_store.similarity_search(
    query = 'Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [15]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599939346313),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.14934504032135)]

In [16]:
# meta-data filtering 
vector_store.similarity_search_with_score(
    query="",
    filter={"team":"Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436006307601929),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909372091293335)]

In [17]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [18]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8af2d290-c2fc-4f7c-984b-4286b321d1f0',
  '239cd839-9e73-4dd7-b268-b77a47bdc413',
  '2cc46849-0f0c-437e-ac9e-581cb4fcabf0',
  'b8c80b6d-283e-4d04-903f-cea8b1babd4c',
  '3e705696-6bc9-4001-9dd9-6b41fe724af0'],
 'embeddings': array([[ 0.00994727,  0.06914337, -0.05147117, ..., -0.03543334,
          0.01284807,  0.0124829 ],
        [ 0.00127744,  0.03129853, -0.02375376, ..., -0.00518358,
         -0.03280612,  0.02737714],
        [-0.10265914,  0.02650812,  0.02271502, ..., -0.03359744,
         -0.07984945, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494375, ..., -0.10995811,
          0.00572558,  0.09915376],
        [ 0.01873978,  0.04382848, -0.04304255, ..., -0.07801619,
         -0.0784068 , -0.00304187]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [19]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [20]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8af2d290-c2fc-4f7c-984b-4286b321d1f0',
  '239cd839-9e73-4dd7-b268-b77a47bdc413',
  '2cc46849-0f0c-437e-ac9e-581cb4fcabf0',
  'b8c80b6d-283e-4d04-903f-cea8b1babd4c',
  '3e705696-6bc9-4001-9dd9-6b41fe724af0'],
 'embeddings': array([[ 0.00994727,  0.06914337, -0.05147117, ..., -0.03543334,
          0.01284807,  0.0124829 ],
        [ 0.00127744,  0.03129853, -0.02375376, ..., -0.00518358,
         -0.03280612,  0.02737714],
        [-0.10265914,  0.02650812,  0.02271502, ..., -0.03359744,
         -0.07984945, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494375, ..., -0.10995811,
          0.00572558,  0.09915376],
        [ 0.01873978,  0.04382848, -0.04304255, ..., -0.07801619,
         -0.0784068 , -0.00304187]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo